# NodeDwarves – Training in Google Colab

This notebook: clones (or updates) the repo, checks Torch/Numpy, runs the training command `ai:train:python:fresh`, and saves outputs to Google Drive.

Update the variables in the next cell if you want a different branch or Drive folder.

In [ ]:
import os

REPO_URL = "https://github.com/Gladiak/nodeDwarves.git"
REPO_DIR = "/content/nodeDwarves"
BRANCH = "main"  # change if needed
DRIVE_DIR = "/content/drive/MyDrive/nodeDwarves_colab"  # change if needed

os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR
os.environ["BRANCH"] = BRANCH
os.environ["DRIVE_DIR"] = DRIVE_DIR

print("REPO_URL:", REPO_URL)
print("REPO_DIR:", REPO_DIR)
print("BRANCH:", BRANCH)
print("DRIVE_DIR:", DRIVE_DIR)

In [ ]:
%%bash
set -euo pipefail

REPO_URL="$REPO_URL"
REPO_DIR="$REPO_DIR"
BRANCH="$BRANCH"

if [ -d "$REPO_DIR/.git" ]; then
  echo "Repo exists. Pulling updates..."
  cd "$REPO_DIR"
  git fetch --all --prune
  git checkout "$BRANCH"
  git pull --ff-only
else
  echo "Cloning repo..."
  git clone "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
  git checkout "$BRANCH"
fi

In [ ]:
import importlib
import platform

def check_pkg(name, import_name=None):
    mod_name = import_name or name
    try:
        module = importlib.import_module(mod_name)
        version = getattr(module, "__version__", "unknown")
        return True, version
    except Exception as exc:
        return False, str(exc)

torch_ok, torch_ver = check_pkg("torch")
numpy_ok, numpy_ver = check_pkg("numpy")

print("Python:", platform.python_version())
print("Torch OK:", torch_ok, "version/info:", torch_ver)
print("Numpy OK:", numpy_ok, "version/info:", numpy_ver)

if torch_ok:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Torch not available. Install if needed before training.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail

cd "$REPO_DIR"
npm run ai:train:python:fresh

In [ ]:
import glob
import os
import shutil
import time

repo_dir = os.environ["REPO_DIR"]
drive_dir = os.environ["DRIVE_DIR"]

run_dirs = sorted(
    glob.glob(os.path.join(repo_dir, "debug", "run_*")),
    key=os.path.getmtime
)

if not run_dirs:
    raise RuntimeError("No debug/run_* directories found. Training may have failed.")

latest_run = run_dirs[-1]
timestamp = time.strftime("%Y%m%d_%H%M%S")
dest_root = os.path.join(drive_dir, f"nodeDwarves_training_{timestamp}")
os.makedirs(dest_root, exist_ok=True)

shutil.copytree(latest_run, os.path.join(dest_root, os.path.basename(latest_run)), dirs_exist_ok=True)

models_dir = os.path.join(repo_dir, "models")
if os.path.isdir(models_dir):
    shutil.copytree(models_dir, os.path.join(dest_root, "models"), dirs_exist_ok=True)

print("Saved training output to:", dest_root)